# Notebook 04: Multimodal Fusion, MC Uncertainty & 5-Arm Feature Ablations

This notebook walks through:
1. Generating a synthetic clinical + spirometry + imaging cohort.
2. Multimodal gated fusion architecture combining CT anomaly embeddings with clinical covariates.
3. Monte Carlo Dropout epistemic and aleatoric uncertainty estimation.
4. 5-arm systematic feature ablation study comparing Clinical, Spirometry, Imaging, and Multimodal models.

> **SYNTHETIC DATA BANNER:** This cohort is synthetically generated for methodological validation only.


In [ ]:
import torch
import pandas as pd
from copd_anomaly.synthetic.cohort_generator import generate_synthetic_copd_cohort, SYNTHETIC_BANNER
from copd_anomaly.synthetic.multimodal import MultimodalRiskPipeline
from scripts.run_ablation import run_ablation_experiments

print(SYNTHETIC_BANNER)


## 1. Inspecting the Synthetic Clinical Cohort


In [ ]:
df_cohort = generate_synthetic_copd_cohort(n_patients=200, seed=42)
display(df_cohort[['patient_id', 'age', 'sex', 'pack_years', 'fev1_pct_predicted', 'gold_stage', 'global_anomaly_score', 'exacerbation_12m_event']].head(10))


## 2. Multimodal Risk Modeling & Uncertainty Estimation


In [ ]:
pipeline = MultimodalRiskPipeline(d_ct=32, d_clin=6, d_fused=16, dropout=0.3)
x_ct_demo = torch.randn(8, 32)
x_clin_demo = torch.randn(8, 6)

pred_dict = pipeline.predict_with_uncertainty(x_ct_demo, x_clin_demo, n_mc_samples=25)
print('Predicted Risk & Uncertainty for Sample Patients:')
for i in range(5):
    print(f'  Patient {i+1}: Mean Risk = {pred_dict["mean_risk"][i]:.1%}, Epistemic Uncertainty = {pred_dict["epistemic_uncertainty"][i]:.4f}')


## 3. Systematic 5-Arm Feature Ablations


In [ ]:
from pathlib import Path
temp_report = Path('../reports/multimodal_ablation_results.csv')
df_ablations = run_ablation_experiments(output_report=temp_report, n_splits=5, seed=42)
display(df_ablations[['configuration', 'auroc', 'auprc', 'brier_score', 'delta_auroc_vs_baseline', 'p_value_vs_baseline']])
